### Setup and Imports
Locate the project root from any notebook working directory, add it to `sys.path`, and import the canonical Phase 1 inspection utilities.


In [1]:
import sys
from pathlib import Path

import pandas as pd
from IPython.display import display


def find_project_root(start: Path) -> Path:
    """Find the repository root from the current notebook working directory."""
    start = start.resolve()
    for candidate in [start, *start.parents]:
        if (candidate / "configs" / "config.yaml").exists() and (candidate / "src").exists():
            return candidate
    raise FileNotFoundError("Could not locate the project root. Run this notebook inside the project directory.")


PROJECT_ROOT = find_project_root(Path.cwd())
if str(PROJECT_ROOT) not in sys.path:
    sys.path.insert(0, str(PROJECT_ROOT))

from src.data_loading import inspect_dataset, load_config, read_csv_sample, resolve_project_path

CONFIG_PATH = PROJECT_ROOT / "configs" / "config.yaml"
config = load_config(CONFIG_PATH)
DATA_PATH = resolve_project_path(config["paths"]["raw_dataset"], PROJECT_ROOT)
METRICS_DIR = resolve_project_path(config["paths"]["metrics_dir"], PROJECT_ROOT)

print("Project root:", PROJECT_ROOT)
print("Dataset path:", DATA_PATH)
print("Metrics directory:", METRICS_DIR)


Project root: E:\Paper Multiclass-Intrusion-Detection-System
Dataset path: E:\Paper Multiclass-Intrusion-Detection-System\data\raw\CIC-ToN-IoT.csv
Metrics directory: E:\Paper Multiclass-Intrusion-Detection-System\results\metrics


### Load a Small Sample
Read only the first 100,000 rows for quick visual inspection. Feature decisions are not made from this sequential sample.


In [2]:
SAMPLE_SIZE = 100_000
sample_df, used_encoding = read_csv_sample(DATA_PATH, SAMPLE_SIZE)

print("Encoding used:", used_encoding)
print("Sample shape:", sample_df.shape)
display(sample_df.head())


Encoding used: utf-8
Sample shape: (100000, 85)


,Flow ID,Src IP,Src Port,Dst IP,Dst Port,Protocol,Timestamp,Flow Duration,Tot Fwd Pkts,Tot Bwd Pkts,...,Active Mean,Active Std,Active Max,Active Min,Idle Mean,Idle Std,Idle Max,Idle Min,Label,Attack
0,177.30.87.144-192.168.1.1-0-0-0,177.30.87.144,0,192.168.1.1,0,0,25/04/2019 05:18:52 pm,47814343,5,0,...,1038036.0,0.000000e+00,1038036.0,1038036.0,5.187256e+14,8.984590e+14,1.556177e+15,1.657324e+07,0,Benign
1,167.49.176.28-50.165.192.168-0-0-0,167.49.176.28,0,50.165.192.168,0,0,25/04/2019 05:18:49 pm,2033142,2,0,...,0.0,0.000000e+00,0.0,0.0,1.556177e+15,0.000000e+00,1.556177e+15,1.556177e+15,0,Benign
2,230.158.52.59-177.21.192.168-0-0-0,230.158.52.59,0,177.21.192.168,0,0,25/04/2019 05:18:37 pm,82877133,14,0,...,1931160.5,1.711593e+06,3942470.0,226402.0,1.729085e+14,5.187256e+14,1.556177e+15,6.036493e+06,0,Benign
3,183.68.192.168-1.1.192.168-0-0-0,183.68.192.168,0,1.1.192.168,0,0,25/04/2019 05:18:42 pm,24359,2,0,...,0.0,0.000000e+00,0.0,0.0,1.556177e+15,0.000000e+00,1.556177e+15,1.556177e+15,0,Benign
4,183.41.192.168-1.1.192.168-0-0-0,183.41.192.168,0,1.1.192.168,0,0,25/04/2019 05:18:42 pm,10239351,3,0,...,4053975.0,0.000000e+00,4053975.0,4053975.0,7.780884e+14,1.100383e+15,1.556177e+15,6.185376e+06,0,Benign


### Inspect Columns and Label Candidates
List raw columns and show likely label columns so the selected target can be checked visually.


In [3]:
label_keywords = ["label", "class", "attack", "category", "type", "target"]
label_candidates = [
    column for column in sample_df.columns
    if any(keyword in column.lower() for keyword in label_keywords)
]

print("Total columns:", len(sample_df.columns))
print("Label candidates:", label_candidates)
for column in label_candidates:
    print("\n" + "=" * 60)
    print("Column:", column)
    print("Unique values in sample:", sample_df[column].nunique(dropna=False))
    print(sample_df[column].value_counts(dropna=False).head(30))


Total columns: 85
Label candidates: ['Label', 'Attack']

Column: Label
Unique values in sample: 2
Label
0    99126
1      874
Name: count, dtype: int64

Column: Attack
Unique values in sample: 7
Attack
Benign       99126
mitm           489
ddos           202
dos            145
scanning        29
injection        7
password         2
Name: count, dtype: int64


### Run Canonical Phase 1 Inspection
Run the reusable chunked inspection from `src.data_loading`. This scans the full dataset and writes canonical outputs to `results/metrics`.


In [4]:
inspection_result = inspect_dataset(CONFIG_PATH, sample_size=100_000, chunk_size=100_000)

print("Phase 1 inspection completed.")
for key, value in inspection_result.items():
    print(f"{key}: {value}")


Phase 1 inspection completed.
dataset_path: E:\Paper Multiclass-Intrusion-Detection-System\data\raw\CIC-ToN-IoT.csv
total_rows: 5351760
total_columns: 85
num_classes: 10
feature_count: 69
drop_count: 15
metrics_dir: E:\Paper Multiclass-Intrusion-Detection-System\results\metrics


### Review Class Distribution
Load the canonical class distribution generated by the inspection script and verify that it matches the proposal totals.


In [5]:
class_distribution = pd.read_csv(METRICS_DIR / "class_distribution.csv")

display(class_distribution)
print("Total rows:", int(class_distribution["count"].sum()))
print("Number of classes:", class_distribution.shape[0])


,class,count,percentage,class_index
0,Benign,2515236,46.998296,0
1,Backdoor,27145,0.507216,1
2,DDoS,202,0.003774,2
3,DoS,145,0.002709,3
4,Injection,277696,5.188872,4
5,MITM,517,0.009660,5
6,Password,340208,6.356937,6
7,Ransomware,5098,0.095258,7
8,Scanning,36205,0.676506,8
9,XSS,2149308,40.160770,9


Total rows: 5351760
Number of classes: 10


### Review Selected Model Features
Load the canonical candidate feature list. After class-conditional review, this project keeps 69 model features.


In [6]:
candidate_features = (METRICS_DIR / "candidate_features.txt").read_text(encoding="utf-8").splitlines()

print("Candidate model features:", len(candidate_features))
for index, feature in enumerate(candidate_features, start=1):
    print(f"{index}. {feature}")


Candidate model features: 69
1. ACK Flag Cnt
2. Active Max
3. Active Mean
4. Active Min
5. Active Std
6. Bwd Blk Rate Avg
7. Bwd Byts/b Avg
8. Bwd Header Len
9. Bwd IAT Max
10. Bwd IAT Mean
11. Bwd IAT Min
12. Bwd IAT Std
13. Bwd IAT Tot
14. Bwd Pkt Len Max
15. Bwd Pkt Len Mean
16. Bwd Pkt Len Min
17. Bwd Pkt Len Std
18. Bwd Pkts/b Avg
19. Bwd Pkts/s
20. Bwd Seg Size Avg
21. Down/Up Ratio
22. Dst Port
23. FIN Flag Cnt
24. Flow Byts/s
25. Flow Duration
26. Flow IAT Max
27. Flow IAT Mean
28. Flow IAT Min
29. Flow IAT Std
30. Flow Pkts/s
31. Fwd Act Data Pkts
32. Fwd Header Len
33. Fwd IAT Max
34. Fwd IAT Mean
35. Fwd IAT Min
36. Fwd IAT Std
37. Fwd IAT Tot
38. Fwd Pkt Len Max
39. Fwd Pkt Len Mean
40. Fwd Pkt Len Min
41. Fwd Pkt Len Std
42. Fwd Pkts/s
43. Fwd PSH Flags
44. Fwd Seg Size Avg
45. Fwd Seg Size Min
46. Idle Max
47. Idle Mean
48. Idle Min
49. Idle Std
50. Init Bwd Win Byts
51. Init Fwd Win Byts
52. PSH Flag Cnt
53. Pkt Len Max
54. Pkt Len Mean
55. Pkt Len Min
56. Pkt Len Std
57

### Review Dropped Features
Check excluded columns and their explicit reasons: identifiers, leakage-prone labels, or empirically quasi-constant features.


In [7]:
dropped_features = pd.read_csv(METRICS_DIR / "dropped_features.csv")

display(dropped_features)
print("Dropped columns:", dropped_features.shape[0])


,feature,recommended_drop_reason
0,Bwd PSH Flags,globally and class-conditionally quasi-constant
1,Bwd URG Flags,globally and class-conditionally quasi-constant
2,CWE Flag Count,globally and class-conditionally quasi-constant
3,Dst IP,identifier or metadata column; non-numeric or ...
4,ECE Flag Cnt,globally and class-conditionally quasi-constant
5,Flow ID,identifier or metadata column; non-numeric or ...
6,Fwd Blk Rate Avg,globally and class-conditionally quasi-constant
7,Fwd Byts/b Avg,globally and class-conditionally quasi-constant
8,Fwd Pkts/b Avg,globally and class-conditionally quasi-constant
9,Fwd URG Flags,globally and class-conditionally quasi-constant


Dropped columns: 15


### Review Quasi-Constant Decisions
Inspect sparse features that need class-conditional review. `Fwd PSH Flags` and `RST Flag Cnt` are retained because they carry minority-class signal.


In [8]:
quasi_constant_review = pd.read_csv(METRICS_DIR / "quasi_constant_review.csv")

display(quasi_constant_review)


,feature,configured_model_feature,configured_drop,full_top_value_frequency,has_class_conditional_signal,recommended_drop_reason
0,Bwd PSH Flags,False,True,1.000000,False,globally and class-conditionally quasi-constant
1,Bwd URG Flags,False,True,1.000000,False,globally and class-conditionally quasi-constant
2,CWE Flag Count,False,True,0.999998,False,globally and class-conditionally quasi-constant
3,ECE Flag Cnt,False,True,0.999998,False,globally and class-conditionally quasi-constant
4,Fwd Blk Rate Avg,False,True,1.000000,False,globally and class-conditionally quasi-constant
5,Fwd Byts/b Avg,False,True,1.000000,False,globally and class-conditionally quasi-constant
6,Fwd Pkts/b Avg,False,True,1.000000,False,globally and class-conditionally quasi-constant
7,Fwd URG Flags,False,True,1.000000,False,globally and class-conditionally quasi-constant
8,Subflow Bwd Pkts,False,True,1.000000,False,globally and class-conditionally quasi-constant
9,URG Flag Cnt,False,True,1.000000,False,globally and class-conditionally quasi-constant


### Inspect Backdoor Signal Features
Show class-conditional statistics for the two sparse features retained after the review.


In [9]:
class_signal_summary = pd.read_csv(METRICS_DIR / "class_signal_summary.csv")
backdoor_signal = class_signal_summary[
    (class_signal_summary["feature"].isin(["Fwd PSH Flags", "RST Flag Cnt"]))
    & (class_signal_summary["class"] == "Backdoor")
]

display(backdoor_signal)


,feature,class,class_index,count,non_zero_count,non_zero_ratio,mean,min,max,unique_count
61,Fwd PSH Flags,Backdoor,1,27145,9874,0.363750,0.363750,0.0,1.0,2
101,RST Flag Cnt,Backdoor,1,27145,16973,0.625272,18.710849,0.0,30.0,22


### Final Phase 1 Checks
Assert the key Phase 1 expectations before moving to preprocessing.


In [10]:
expected_rows = config["data"]["expected_rows"]
expected_columns = config["data"]["expected_total_columns"]
expected_classes = config["data"]["expected_num_classes"]
expected_features = config["data"]["expected_feature_count"]

assert inspection_result["total_rows"] == expected_rows
assert inspection_result["total_columns"] == expected_columns
assert inspection_result["num_classes"] == expected_classes
assert len(candidate_features) == expected_features
assert "Fwd PSH Flags" in candidate_features
assert "RST Flag Cnt" in candidate_features

print("Phase 1 notebook checks passed.")


Phase 1 notebook checks passed.
